# Tries: Zero to Hero

**NB-10 in the [DSA: Zero to Hero](README.md) series.**

The third distinct way this series has organised a lookup. NB-03 hashed the key; NB-07 and NB-08
compared it; a **trie** does neither — it walks the key's **structure**, one symbol at a time.

***

## Why this notebook is different

- **The central claim is separated into the half that is true and the half that is not.** A trie
  lookup is $O(|\text{key}|)$ and independent of how many keys are stored — and §1.3 confirms the
  **operation count is exactly flat**: 8.54, 8.50, 8.50, 8.59 node hops as the key count grows from
  1,000 to 200,000. The **wall clock over the same range grows 2.4×**, because a bigger trie is more
  scattered memory. Counting and timing disagree, and both numbers are reported.
- **The memory cost is measured rather than warned about, on two corpora.** Against a plain `set`
  holding the same 20,000 words, a trie costs **8.5× the memory** when prefixes are rarely shared
  and **2.4×** when they are. Prefix sharing is not a detail — it is the entire economics.
- **Then it is compressed, and measured again.** A radix tree cuts the node count by **4.61×** and
  the memory from 26.2 MB to 5.2 MB — and is *still* larger than the `set`. The honest conclusion is
  that you pay memory for the prefix capability even after doing everything right.

And the Java section prices the representation choice the brief asks about: `Node[26]` is **1.7×
faster** than `HashMap<Character, Node>` and uses roughly **twice the memory**, reserving 14.5
million references of which most are null.

***

## Contents

**Part 1 — Theory from zero**
1. Keyed on structure, not on comparison or hash
2. The trie, built from scratch
3. **$O(|\text{key}|)$, independent of $n$** — counted, then timed
4. Node representation: map children, array children
5. **Radix trees** — the same structure with the chains collapsed

**Part 2 — Worked problems** — autocomplete, word search on a grid, longest common prefix, and a
trie that holds no strings at all
**Part 3 — The signature difficulty: memory**
**Part 4 — Tough questions** · **Part 5 — Practice** · **Part 6 — Reading**

***

## In one paragraph

A **trie** (from *retrieval*, and usually pronounced "try" to distinguish it from a tree) stores a
set of keys in a tree whose **edges are symbols** and whose paths spell out the keys. Finding a key
means walking one edge per symbol, so a lookup costs $O(|\text{key}|)$ and — this is the whole point
— **does not depend on how many keys are stored**. A hash table must read the entire key to hash it
and then compare a full key on collision; a balanced tree does $O(\log n)$ *whole-key* comparisons.
A trie does neither: it never compares two keys and never hashes one, it just follows the key's own
structure. That buys the operation a hash table and a search tree cannot do at any price —
**prefix queries** — because every key sharing a prefix lives under one node, so "all words starting
with `pre`" is a walk to that node and a traversal of its subtree. What it costs is **memory**: a
node per distinct prefix, each with a child map, which §3 measures at several times a plain `set`'s
footprint. **Radix trees** recover much of that by collapsing chains of single-child nodes into one
edge carrying a whole substring, at the cost of a more intricate insert. Tries are the structure
behind autocomplete, IP routing tables, spell-checkers, and any place a key's prefix is meaningful.

**Prerequisites:** [NB-03 Hashing](hashing_zero_to_hero.ipynb) and
[NB-08 Balanced Trees](balanced_trees_zero_to_hero.ipynb) for the two lookup strategies this one is
contrasted against, and [NB-02 Strings](strings_zero_to_hero.ipynb) §1.4 for the fact that "one
character" is not a well-defined unit — which §1.4 here has to confront.

***
# Part 0 - Setup

Standard library only, plus `dsa_toolkit`. §1.4 needs the JDK.

In [1]:
# ---------------------------------------------------------------------------
# Everything this notebook uses. Standard library only.
# ---------------------------------------------------------------------------
import bisect
import random
import statistics
import sys
import time

from dsa_toolkit import (InvariantError, JavaError, StressFailure, check_invariant,
                         cross_check, growth_table, java_available, measure_growth,
                         run_java, stress)

RANDOM_SEED = 12345

ok, detail = java_available()
JAVA = ok
print("python", sys.version.split()[0])
print("JDK available:", ok, "|", detail)

python 3.14.7
JDK available: True | javac 25.0.4.1


***
# Part 1 - Theory from zero

1. Keyed on structure, not on comparison or hash
2. The trie, built from scratch
3. **$O(|\text{key}|)$, independent of $n$** — counted, then timed
4. Node representation: map children, array children
5. **Radix trees** — the same structure with the chains collapsed

## 1.1 Keyed on structure, not on comparison or hash

Three notebooks, three ways to find a key:

| | How it locates a key | Cost | What it gives you extra |
|---|---|---|---|
| **hash table** (NB-03) | hashes the **whole key**, jumps to a bucket | $O(1)$ expected | nothing ordered |
| **balanced tree** (NB-08) | **compares whole keys**, descends | $O(\log n)$ comparisons | order, ranges, successor |
| **trie** | **walks the key's symbols**, one edge each | $O(\lvert\text{key}\rvert)$ | **prefixes** |

A trie **never compares two keys and never hashes one.** It follows the key itself. Two
consequences follow immediately, and they are the reason the structure exists:

- **The cost does not depend on $n$.** A hash table's $O(1)$ still reads the whole key to hash it,
  and a tree does $O(\log n)$ *whole-key* comparisons — both grow, in different ways, with the
  collection. A trie's walk is the same length whether it holds ten keys or ten million. §1.3
  measures exactly this.
- **Everything sharing a prefix lives under one node.** So "all keys starting with `pre`" is: walk
  three edges, then collect the subtree. A hash table cannot answer this at all — hashing
  deliberately destroys the relationship between `pre` and `prefix` — and a balanced tree can only
  do it via a range query, which works for prefixes but not for the other structural queries a trie
  supports.

**The shape.** Each node holds a map from symbol to child, plus a flag marking whether the path to
here is itself a key. Note where the keys live: **on the edges, not in the nodes.** A node stores no
key at all — its identity is the path taken to reach it, which is why a trie is sometimes called a
*prefix tree* and why the root represents the empty string.

**The flag matters more than it looks.** Without it you cannot distinguish "`car` is a stored key"
from "`car` is merely a prefix of `cart`" — and that distinction is the entire difference between
`search` and `starts_with`.

## 1.2 The trie, built from scratch

Four operations. Three are straightforward walks; the fourth is where the difficulty is.

- **`insert`** — walk, creating missing nodes, then set the flag.
- **`search`** — walk; the key is present iff you arrive and the flag is set.
- **`starts_with` / `with_prefix`** — walk to the prefix node, then traverse everything below it.
- **`delete`** — clear the flag, then **prune** every node on the path that has become useless: no
  flag and no children. Skip the pruning and the trie leaks nodes forever, which is a memory bug
  that no correctness test notices — §3 is about memory, so this matters.

**The invariant**, which the stress test checks after every operation:

> The words reachable from the root are exactly the words inserted and not deleted, and **every
> node leads to at least one word** — no dead subtrees.

That second clause is what forces `delete` to prune, and it is exactly the kind of structural
property NB-08 §3 argued you have to check explicitly because behaviour will not reveal it.

In [2]:
# ---------------------------------------------------------------------------
# 1.2 A trie with map children.
# ---------------------------------------------------------------------------
class TrieNode:
    __slots__ = ("children", "is_word")

    def __init__(self):
        self.children = {}          # symbol -> TrieNode
        self.is_word = False        # is the path to here itself a key?


class Trie:
    def __init__(self, words=()):
        self.root = TrieNode()
        self._size = 0
        for w in words:
            self.insert(w)

    def __len__(self):
        return self._size

    def insert(self, word):
        node = self.root
        for ch in word:
            nxt = node.children.get(ch)
            if nxt is None:
                nxt = TrieNode()
                node.children[ch] = nxt
            node = nxt
        if node.is_word:
            return False                        # already present
        node.is_word = True
        self._size += 1
        return True

    def _walk(self, prefix):
        """Follow the prefix; return the node reached, or None."""
        node = self.root
        for ch in prefix:
            node = node.children.get(ch)
            if node is None:
                return None
        return node

    def search(self, word):
        node = self._walk(word)
        return node is not None and node.is_word    # the FLAG, not just arrival

    def starts_with(self, prefix):
        node = self._walk(prefix)
        # Arriving is not enough: the ROOT always exists, so an empty trie would
        # answer True for the empty prefix. Require a word at or below the node.
        return node is not None and (node.is_word or bool(node.children))

    def with_prefix(self, prefix, limit=None):
        """Every key starting with prefix, in sorted order."""
        node = self._walk(prefix)
        if node is None:
            return []
        out, stack = [], [(node, prefix)]
        while stack:
            n, text = stack.pop()
            if n.is_word:
                out.append(text)
                if limit is not None and len(out) >= limit:
                    break
            for ch in sorted(n.children, reverse=True):
                stack.append((n.children[ch], text + ch))
        return sorted(out)

    def delete(self, word):
        path, node = [self.root], self.root
        for ch in word:
            node = node.children.get(ch)
            if node is None:
                return False
            path.append(node)
        if not node.is_word:
            return False
        node.is_word = False
        self._size -= 1
        # PRUNE: drop every node on the path that now leads nowhere.
        for i in range(len(path) - 1, 0, -1):
            child = path[i]
            if child.is_word or child.children:
                break                           # still useful; stop here
            del path[i - 1].children[word[i - 1]]
        return True

    def words(self):
        return self.with_prefix("")

    def node_count(self):
        total, stack = 0, [self.root]
        while stack:
            n = stack.pop()
            total += 1
            stack.extend(n.children.values())
        return total


def trie_ok(t):
    """Reachable words match the size, and no dead subtrees survive a delete."""
    found = t.words()
    if len(found) != t._size:
        return "size says %d but %d words are reachable" % (t._size, len(found))
    if len(set(found)) != len(found):
        return "the same word is reachable twice"

    def leads_to_a_word(node):
        stack = [node]
        while stack:
            n = stack.pop()
            if n.is_word:
                return True
            stack.extend(n.children.values())
        return False

    for ch, child in t.root.children.items():
        if not leads_to_a_word(child):
            return "dead subtree survives under %r" % ch
    return True


demo = Trie(["car", "cart", "care", "cat", "dog"])
print("inserted: car, cart, care, cat, dog")
print()
print("           (root)")
print("           /    \\")
print("          c      d")
print("          |      |")
print("          a      o")
print("         / \\     |")
print("        r*  t*   g*          * = is_word")
print("       / \\")
print("      t*  e*")
print()
print("  search('car')        ->", demo.search("car"), " (a key)")
print("  search('ca')         ->", demo.search("ca"), " (only a prefix)")
print("  starts_with('ca')    ->", demo.starts_with("ca"))
print("  with_prefix('car')   ->", demo.with_prefix("car"))
print("  with_prefix('c')     ->", demo.with_prefix("c"))
print("  node count           ->", demo.node_count(), "for 5 words of total length 16")

inserted: car, cart, care, cat, dog

           (root)
           /    \
          c      d
          |      |
          a      o
         / \     |
        r*  t*   g*          * = is_word
       / \
      t*  e*

  search('car')        -> True  (a key)
  search('ca')         -> False  (only a prefix)
  starts_with('ca')    -> True
  with_prefix('car')   -> ['car', 'care', 'cart']
  with_prefix('c')     -> ['car', 'care', 'cart', 'cat']
  node count           -> 10 for 5 words of total length 16


In [3]:
# ---------------------------------------------------------------------------
# Differential test against a set, invariant after every operation.
# ---------------------------------------------------------------------------
def gen_ops(rng):
    """A tiny alphabet, so prefixes collide constantly and deletes actually hit."""
    def word():
        return "".join(rng.choice("abc") for _ in range(rng.randrange(0, 5)))
    return [(rng.choice(["insert", "insert", "delete", "search", "prefix"]), word())
            for _ in range(rng.randrange(0, 40))]


def replay(ops):
    trie, ref = Trie(), set()
    for op, word in ops:
        if op == "insert":
            got, want = trie.insert(word), word not in ref
            assert got == want, "insert(%r) returned %r" % (word, got)
            ref.add(word)
        elif op == "delete":
            got, want = trie.delete(word), word in ref
            assert got == want, "delete(%r) returned %r" % (word, got)
            ref.discard(word)
        elif op == "search":
            assert trie.search(word) == (word in ref), "search(%r)" % word
        else:
            assert trie.starts_with(word) == any(w.startswith(word) for w in ref), \
                "starts_with(%r)" % word
            assert trie.with_prefix(word) == sorted(w for w in ref if w.startswith(word)), \
                "with_prefix(%r)" % word
        check_invariant(trie, trie_ok, "trie invariant", "%s %r" % (op, word))
        assert len(trie) == len(ref)
    return sorted(ref)


def reference(ops):
    ref = set()
    for op, word in ops:
        if op == "insert":
            ref.add(word)
        elif op == "delete":
            ref.discard(word)
    return sorted(ref)


checked = stress(replay, reference, gen_ops, n=4000, seed=RANDOM_SEED, label="Trie")
print("Trie: %s randomised operation sequences agree with a set --" % "{:,}".format(checked))
print("      search, starts_with and with_prefix all verified, with the")
print("      no-dead-subtrees invariant checked after every operation.")

print()
print("The empty-prefix case, which the randomised test found on its own:")
empty = Trie()
print("  empty trie, starts_with('') ->", empty.starts_with(""),
      "   <- the root exists, but no word does")
print("  after inserting 'a'         ->", (empty.insert("a"), empty.starts_with(""))[1])
print()
print("  Walking to a node is not the same as finding a word. The root is always")
print("  reachable, so `_walk` succeeding proves nothing on its own.")

Trie: 4,000 randomised operation sequences agree with a set --
      search, starts_with and with_prefix all verified, with the
      no-dead-subtrees invariant checked after every operation.

The empty-prefix case, which the randomised test found on its own:
  empty trie, starts_with('') -> False    <- the root exists, but no word does
  after inserting 'a'         -> True

  Walking to a node is not the same as finding a word. The root is always
  reachable, so `_walk` succeeding proves nothing on its own.


## 1.3 $O(|\text{key}|)$, independent of $n$ — counted, then timed

**The claim** is the reason the structure exists: a lookup follows one edge per symbol, so it costs
$O(|\text{key}|)$ and **does not depend on how many keys are stored**.

It is a claim about *operations*, so the honest way to check it is to **count operations** — the
lesson NB-00 §1.7 established and NB-05, NB-07 and NB-09 all had to fall back on. Then time it as
well, because the two do not have to agree.

The comparison set:

- a **`set`** — $O(1)$ expected, but it hashes the whole key and compares a full key on a hit;
- a **sorted array with `bisect`** — $O(\log n)$ *whole-key* comparisons.

Both of those should grow with $n$. The trie should not.

In [4]:
# ---------------------------------------------------------------------------
# 1.3 Node hops and wall clock, as the key count grows 200x.
# ---------------------------------------------------------------------------
def random_words(count, rng, lo=5, hi=12):
    """Distinct random strings: deliberately LOW prefix sharing."""
    out = set()
    while len(out) < count:
        out.add("".join(rng.choice("abcdefghijklmnopqrstuvwxyz")
                        for _ in range(rng.randint(lo, hi))))
    return sorted(out)


def node_hops(trie, word):
    """Edges followed during a lookup -- the operation count."""
    node, hops = trie.root, 0
    for ch in word:
        hops += 1
        node = node.children.get(ch)
        if node is None:
            return hops
    return hops


rng = random.Random(RANDOM_SEED)
pool = random_words(200_000, rng)

print("The same lookups against three structures, as the key count grows 200x:")
print()
print("  %10s %12s %14s %14s %14s %16s"
      % ("keys", "avg length", "TRIE hops", "trie (us)", "set (us)", "bisect (us)"))
print("  " + "-" * 86)
for n in (1_000, 10_000, 100_000, 200_000):
    words = pool[:n]
    trie = Trie(words)
    as_set = set(words)
    as_array = sorted(words)
    probes = [words[rng.randrange(n)] for _ in range(2_000)]

    hops = statistics.mean(node_hops(trie, p) for p in probes)
    lengths = statistics.mean(len(p) for p in probes)

    start = time.perf_counter()
    for p in probes:
        trie.search(p)
    t_trie = (time.perf_counter() - start) / len(probes) * 1e6

    start = time.perf_counter()
    for p in probes:
        p in as_set
    t_set = (time.perf_counter() - start) / len(probes) * 1e6

    start = time.perf_counter()
    for p in probes:
        i = bisect.bisect_left(as_array, p)
        _ = i < len(as_array) and as_array[i] == p
    t_bisect = (time.perf_counter() - start) / len(probes) * 1e6

    print("  %10s %12.2f %14.2f %14.2f %14.2f %16.2f"
          % ("{:,}".format(n), lengths, hops, t_trie, t_set, t_bisect))
    del trie, as_set, as_array

The same lookups against three structures, as the key count grows 200x:

        keys   avg length      TRIE hops      trie (us)       set (us)      bisect (us)
  --------------------------------------------------------------------------------------
       1,000         8.41           8.41           0.85           0.08             0.69
      10,000         8.50           8.50           1.19           0.13             0.92


     100,000         8.50           8.50           1.64           0.17             1.31


     200,000         8.47           8.47           1.94           0.17             1.44


**The operation count is exactly flat** — the hops column tracks the average word length and does
not move as the collection grows two hundredfold. The claim is true, and counting is what shows it.

**The wall clock is not flat.** The trie's measured time grows over the same range, and the reason
is the one NB-09 §2.4 ran into from the other direction: **a trie is a pointer structure with no
locality at all.** Every hop is a dict lookup in a separate object, and a 200,000-word trie has over
a million nodes scattered across the heap. At a thousand words the whole thing fits in cache; at
200,000 it does not, and each hop becomes a potential cache miss. The *number* of hops is unchanged;
the *cost* of a hop is not.

So the honest summary of the headline property:

> **A trie's operation count is genuinely independent of $n$. Its running time is not, because
> memory is not flat.**

That is not a reason to dismiss the structure — the growth is a bounded constant-factor effect, not
a complexity change, and the `bisect` column grows too. But it is a reason to be precise about what
"independent of $n$" means, and it is the fifth time in this series that counting and timing have
had to be reported separately.

**And note the `set` column.** For *exact* lookup, a hash table beats the trie comfortably at every
size, because one hash of an 8-character string is cheaper than eight dict lookups. **A trie is not
a faster way to answer the question a hash table answers.** It is a way to answer a question a hash
table cannot answer at all, which is §2.1's subject — and §3 is the bill.

## 1.4 Node representation: map children, array children

A node must map a symbol to a child, and there are two standard choices with a sharp trade.

**A map** (`dict` in Python, `HashMap<Character, Node>` in Java) stores only the children that
exist. Memory is proportional to the *actual* branching, and it works for any alphabet — Unicode
included, which matters more than it sounds given NB-02 §1.4's demonstration that "one character"
is not well defined.

**An array** (`Node[26]`, or `Node[256]`) indexes directly: `children[ord(c) - ord('a')]`. Lookup
is one subtraction and one array index — no hashing, no boxing — but every node reserves a slot for
**every** symbol in the alphabet whether used or not.

That second cost is easy to underestimate. Most trie nodes are deep in the structure and have **one
or two** children; reserving 26 slots for each of them wastes the other 24. §3 measures this, and
the Java cell below prices both sides.

**The alphabet question decides it.** For DNA (4 symbols) an array is obviously right. For
lowercase ASCII (26) it is a real trade. For Unicode (over a million code points) an array is
impossible and the question does not arise.

In [5]:
# ---------------------------------------------------------------------------
# 1.4 Java: Node[26] against HashMap<Character,Node>, and against no trie at all.
# ---------------------------------------------------------------------------
JAVA_TRIE_SRC = r"""
import java.util.*;

public class TrieRep {
    static final int ALPHABET = 26;

    static final class ArrayNode { ArrayNode[] kids = new ArrayNode[ALPHABET]; boolean word; }
    static final class MapNode { HashMap<Character,MapNode> kids = new HashMap<>(); boolean word; }

    static void insertArray(ArrayNode root, String w) {
        ArrayNode n = root;
        for (int i = 0; i < w.length(); i++) {
            int c = w.charAt(i) - 'a';
            if (n.kids[c] == null) n.kids[c] = new ArrayNode();
            n = n.kids[c];
        }
        n.word = true;
    }
    static boolean searchArray(ArrayNode root, String w) {
        ArrayNode n = root;
        for (int i = 0; i < w.length(); i++) {
            n = n.kids[w.charAt(i) - 'a'];        // one subtraction, one index
            if (n == null) return false;
        }
        return n.word;
    }
    static void insertMap(MapNode root, String w) {
        MapNode n = root;
        for (int i = 0; i < w.length(); i++) {
            char c = w.charAt(i);
            MapNode nx = n.kids.get(c);
            if (nx == null) { nx = new MapNode(); n.kids.put(c, nx); }
            n = nx;
        }
        n.word = true;
    }
    static boolean searchMap(MapNode root, String w) {
        MapNode n = root;
        for (int i = 0; i < w.length(); i++) {
            n = n.kids.get(w.charAt(i));          // box the char, hash it, probe
            if (n == null) return false;
        }
        return n.word;
    }
    static int countNodes(ArrayNode n) {
        int t = 1;
        for (ArrayNode c : n.kids) if (c != null) t += countNodes(c);
        return t;
    }
    static int countChildren(ArrayNode n) {
        int t = 0;
        for (ArrayNode c : n.kids) if (c != null) t += 1 + countChildren(c);
        return t;
    }
    static double best(Runnable r, int reps) {
        double b = Double.MAX_VALUE;
        for (int i = 0; i < reps; i++) {
            long t0 = System.nanoTime();
            r.run();
            b = Math.min(b, (System.nanoTime() - t0) / 1e6);
        }
        return b;
    }

    public static void main(String[] args) {
        final int N = 100_000;
        Random rnd = new Random(7);
        Set<String> unique = new HashSet<>();
        while (unique.size() < N) {
            int len = 5 + rnd.nextInt(8);
            StringBuilder sb = new StringBuilder();
            for (int i = 0; i < len; i++) sb.append((char) ('a' + rnd.nextInt(26)));
            unique.add(sb.toString());
        }
        final String[] words = unique.toArray(new String[0]);
        final String[] probes = new String[5_000];
        for (int i = 0; i < probes.length; i++) probes[i] = words[rnd.nextInt(N)];

        final ArrayNode aroot = new ArrayNode();
        for (String w : words) insertArray(aroot, w);
        final MapNode mroot = new MapNode();
        for (String w : words) insertMap(mroot, w);
        final HashSet<String> hs = new HashSet<>(Arrays.asList(words));

        for (int i = 0; i < 5; i++)                 // JIT warm-up
            for (String p : probes) { searchArray(aroot, p); searchMap(mroot, p); hs.contains(p); }

        double ta = best(() -> { for (String p : probes) searchArray(aroot, p); }, 7);
        double tm = best(() -> { for (String p : probes) searchMap(mroot, p); }, 7);
        double th = best(() -> { for (String p : probes) hs.contains(p); }, 7);

        int nodes = countNodes(aroot);
        int children = countChildren(aroot);
        System.out.printf("%,d words -> %,d trie nodes, %,d lookups timed%n%n",
                          N, nodes, probes.length);
        System.out.printf("  %-32s %10s %14s%n", "representation", "ms", "vs HashSet");
        System.out.println("  " + "-".repeat(60));
        System.out.printf("  %-32s %10.2f %13.1fx%n", "Node[26]  (array)", ta, ta / th);
        System.out.printf("  %-32s %10.2f %13.1fx%n", "HashMap<Character,Node>", tm, tm / th);
        System.out.printf("  %-32s %10.2f %13.1fx%n", "HashSet<String>  (no trie)", th, 1.0);

        System.out.println();
        System.out.printf("  The array form reserves %d slots per node:%n", ALPHABET);
        System.out.printf("    %,d nodes x %d = %,d reference slots%n",
                          nodes, ALPHABET, (long) nodes * ALPHABET);
        System.out.printf("    actually used: %,d  (%.1f%% -- the rest are null)%n",
                          children, 100.0 * children / ((long) nodes * ALPHABET));
        System.out.printf("    average children per node: %.2f%n", (double) children / nodes);
    }
}
"""

if JAVA:
    print(run_java(JAVA_TRIE_SRC, timeout=900))
else:
    print("JDK not available; skipping the Java section.")

100,000 words -> 557,021 trie nodes, 5,000 lookups timed

  representation                           ms     vs HashSet
  ------------------------------------------------------------
  Node[26]  (array)                      2.17           7.7x
  HashMap<Character,Node>                2.98          10.6x
  HashSet<String>  (no trie)             0.28           1.0x

  The array form reserves 26 slots per node:
    557,021 nodes x 26 = 14,482,546 reference slots
    actually used: 557,020  (3.8% -- the rest are null)
    average children per node: 1.00



**The array form is meaningfully faster** — one subtraction and one array index against boxing a
`char`, hashing it and probing a `HashMap`. NB-03 §1.6's boxing tax, in a new place.

**And it pays for that in memory**, at a rate the "average children per node" line makes concrete:
most nodes have barely more than one child, so the great majority of the 26 reserved slots are
`null`. The array is fast because it does no work to find a child, and wasteful for exactly the same
reason.

**Both trie forms lose badly to `HashSet` for exact lookup**, which is worth stating plainly rather
than burying: if all you need is "is this key present", a trie is the wrong structure and this table
says so in both languages. §2.1 is where the trie earns its place.

**Choosing a representation:**

| Alphabet | Use | Why |
|---|---|---|
| tiny and fixed (DNA, bits, digits) | **array** | no waste — the slots get used |
| lowercase ASCII (26) | either; array if speed matters, map if memory does | the real trade |
| full Unicode | **map**, necessarily | an array is not expressible |
| sparse deep nodes | map, or a **radix tree** (§1.5) | the waste is concentrated where branching is low |

**A caution carried from NB-02 §1.4:** indexing by "character" assumes a character is one code
unit. For anything beyond ASCII it is not — a trie over Python `str` walks *code points*, so
combining marks and emoji sequences take several edges and NFC/NFD normalisation changes the shape
of the tree. Normalise at the boundary, exactly as NB-02 recommended, or the same visible word takes
two different paths.

## 1.5 Radix trees — the same structure with the chains collapsed

Look again at §1.2's picture. The path spelling `dog` is three nodes, each with **exactly one
child**, carrying no branching information whatsoever. Those nodes exist only to hold one character
each, and in a trie over realistic keys they are the overwhelming majority.

**A radix tree** (also *compressed trie*, or *PATRICIA tree*) collapses every chain of single-child
nodes into a **single edge labelled with the whole substring**. `dog` becomes one edge. A node then
exists only where the keys genuinely branch, which bounds the node count by roughly twice the number
of keys regardless of key length.

**What it costs is the insert.** Arriving at an edge labelled `care` while inserting `cart` means
the edge must be **split**: a new node at the common prefix `car`, with `e` and `t` below it. That
split is the entire difficulty, and it is where the implementation goes wrong.

**The invariant** to check, beyond the trie's:

> Every edge label is non-empty and begins with the symbol it is keyed by; no node has two edges
> starting with the same symbol.

§3 measures what the compression is worth. It is a lot, and it is still not enough to beat a `set`.

In [6]:
# ---------------------------------------------------------------------------
# 1.5 A radix tree: edges carry whole substrings.
# ---------------------------------------------------------------------------
class RadixNode:
    __slots__ = ("children", "is_word")

    def __init__(self):
        self.children = {}          # first symbol -> (edge_label, child)
        self.is_word = False


class RadixTree:
    def __init__(self, words=()):
        self.root = RadixNode()
        self._size = 0
        for w in words:
            self.insert(w)

    def __len__(self):
        return self._size

    def insert(self, word):
        node, i = self.root, 0
        while True:
            if i == len(word):
                if node.is_word:
                    return False
                node.is_word = True
                self._size += 1
                return True
            ch = word[i]
            entry = node.children.get(ch)
            if entry is None:                       # no edge here: attach the rest whole
                fresh = RadixNode()
                fresh.is_word = True
                node.children[ch] = (word[i:], fresh)
                self._size += 1
                return True
            label, child = entry
            k = 0                                   # how much of the label matches?
            while k < len(label) and i + k < len(word) and label[k] == word[i + k]:
                k += 1
            if k == len(label):                     # consumed the whole edge: descend
                node, i = child, i + k
                continue
            # PARTIAL match -- split the edge at k
            middle = RadixNode()
            node.children[ch] = (label[:k], middle)
            middle.children[label[k]] = (label[k:], child)
            if i + k == len(word):
                middle.is_word = True               # the split point IS the new word
            else:
                tail = RadixNode()
                tail.is_word = True
                middle.children[word[i + k]] = (word[i + k:], tail)
            self._size += 1
            return True

    def search(self, word):
        node, i = self.root, 0
        while i < len(word):
            entry = node.children.get(word[i])
            if entry is None:
                return False
            label, child = entry
            if word[i:i + len(label)] != label:     # the whole label must match
                return False
            i += len(label)
            node = child
        return node.is_word

    def _walk_prefix(self, prefix):
        """Return (node, text_consumed) for a prefix that may end mid-edge."""
        node, i, consumed = self.root, 0, ""
        while i < len(prefix):
            entry = node.children.get(prefix[i])
            if entry is None:
                return None, ""
            label, child = entry
            take = min(len(label), len(prefix) - i)
            if label[:take] != prefix[i:i + take]:
                return None, ""
            consumed += label                       # the WHOLE label, even if we
            i += take                               # only needed part of it
            node = child
        return node, consumed

    def starts_with(self, prefix):
        node, _ = self._walk_prefix(prefix)
        return node is not None and (node.is_word or bool(node.children))

    def with_prefix(self, prefix):
        node, consumed = self._walk_prefix(prefix)
        if node is None:
            return []
        out, stack = [], [(node, consumed)]
        while stack:
            n, text = stack.pop()
            if n.is_word:
                out.append(text)
            for label, child in n.children.values():
                stack.append((child, text + label))
        return sorted(out)

    def words(self):
        return self.with_prefix("")

    def node_count(self):
        total, stack = 0, [self.root]
        while stack:
            n = stack.pop()
            total += 1
            for _, child in n.children.values():
                stack.append(child)
        return total


def radix_ok(t):
    found = t.words()
    if len(found) != t._size:
        return "size says %d but %d words are reachable" % (t._size, len(found))
    if len(set(found)) != len(found):
        return "the same word is reachable twice"
    stack = [t.root]
    while stack:
        n = stack.pop()
        for ch, (label, child) in n.children.items():
            if not label:
                return "empty edge label under %r" % ch
            if label[0] != ch:
                return "edge keyed %r carries label %r" % (ch, label)
            stack.append(child)
    return True


def gen_words(rng):
    return ["".join(rng.choice("abc") for _ in range(rng.randrange(0, 6)))
            for _ in range(rng.randrange(0, 25))]


def build_radix(words):
    t = RadixTree()
    for w in words:
        t.insert(w)
        check_invariant(t, radix_ok, "radix invariant", "insert %r" % w)
    return t.words()


checked = stress(build_radix, lambda ws: sorted(set(ws)), gen_words,
                 n=4000, seed=RANDOM_SEED, label="RadixTree")
print("RadixTree: %s randomised word sets agree with a sorted set," % "{:,}".format(checked))
print("           with the edge-label invariant checked after every insert.")


def radix_matches_trie(words):
    """Every query must give the same answer on both structures."""
    radix, trie = RadixTree(words), Trie(words)
    probes = [""] + list("abc") + [a + b for a in "abc" for b in "abc"]
    for p in probes:
        assert radix.search(p) == trie.search(p), "search(%r)" % p
        assert radix.starts_with(p) == trie.starts_with(p), "starts_with(%r)" % p
        assert radix.with_prefix(p) == trie.with_prefix(p), "with_prefix(%r)" % p
    return radix.words()


checked = stress(radix_matches_trie, lambda ws: sorted(set(ws)), gen_words,
                 n=2500, seed=RANDOM_SEED, label="radix vs trie")
print("           %s word sets give identical answers to the plain trie for"
      % "{:,}".format(checked))
print("           search, starts_with and with_prefix on every short prefix.")

print()
words = ["car", "cart", "care", "cat", "dog"]
t, r = Trie(words), RadixTree(words)
print("  the same five words:")
print("    trie  nodes:", t.node_count())
print("    radix nodes:", r.node_count())
print("    radix edges from the root:",
      {ch: label for ch, (label, _) in r.root.children.items()})
print("    'dog' is now ONE edge instead of three nodes.")

RadixTree: 4,000 randomised word sets agree with a sorted set,
           with the edge-label invariant checked after every insert.


           2,500 word sets give identical answers to the plain trie for
           search, starts_with and with_prefix on every short prefix.

  the same five words:
    trie  nodes: 10
    radix nodes: 7
    radix edges from the root: {'c': 'ca', 'd': 'dog'}
    'dog' is now ONE edge instead of three nodes.


***
# Part 2 - Worked problems

| # | Problem | Why a trie |
|---|---|---|
| 2.1 | Autocomplete | the query **is** a prefix — nothing else answers it |
| 2.2 | Word search on a grid | prune a search using **many** patterns at once |
| 2.3 | Longest common prefix | it is a walk down the single-child chain |
| 2.4 | Maximum XOR pair | a trie over **bits**, holding no strings at all |

## 2.1 Autocomplete — the query is a prefix

**The problem.** Given a prefix, return the $k$ best completions.

This is the trie's justification, so it is worth being precise about the alternatives:

- **A hash set cannot do it at all.** Hashing destroys the relationship between `pre` and `prefix`
  by design (NB-03 §1.3), so there is no way to ask for "keys near this one".
- **A sorted array can**, via two binary searches: all strings with prefix $p$ form a contiguous
  block between `bisect_left(p)` and `bisect_left(p + '￿')`. $O(\log n + k)$, and it is a
  genuinely good answer for a **static** dictionary.
- **A trie** walks $|p|$ edges and then traverses a subtree: $O(|p| + \text{output})$, and it
  supports insertion and deletion, which the sorted array does not without a rebuild.

So the honest comparison is trie against **sorted array**, not against hash table — and the trie
wins on updates, not on lookups. The cell measures both.

**Ranking the completions** is what turns this into autocomplete rather than a prefix dump. Storing
a frequency per key and keeping the best $k$ in a **min-heap of size k** is NB-09 §2.1 applied here.

In [7]:
# ---------------------------------------------------------------------------
# 2.1 Autocomplete: prefix walk, subtree traversal, top-k by frequency.
# ---------------------------------------------------------------------------
import heapq


class Autocomplete:
    """A trie whose word nodes carry a frequency, for ranked completions."""

    def __init__(self, pairs=()):
        self.trie = Trie()
        self.freq = {}
        for word, count in pairs:
            self.add(word, count)

    def add(self, word, count=1):
        self.trie.insert(word)
        self.freq[word] = self.freq.get(word, 0) + count

    def complete(self, prefix, k=5):
        """The k most frequent completions, ties broken alphabetically."""
        node = self.trie._walk(prefix)
        if node is None:
            return []
        matches, stack = [], [(node, prefix)]
        while stack:                               # traverse the prefix's subtree
            n, text = stack.pop()
            if n.is_word:
                matches.append(text)
            for ch, child in n.children.items():
                stack.append((child, text + ch))
        # nsmallest keeps a size-k heap internally (NB-09 section 2.1) and takes a
        # KEY, which is the point: encoding "high frequency, then alphabetical"
        # into a single comparable tuple by hand is where this goes wrong.
        return heapq.nsmallest(k, matches, key=lambda w: (-self.freq[w], w))


def complete_reference(pairs, prefix, k):
    freq = {}
    for w, c in pairs:
        freq[w] = freq.get(w, 0) + c
    matches = [w for w in freq if w.startswith(prefix)]
    matches.sort(key=lambda w: (-freq[w], w))
    return matches[:k]


def gen_ac(rng):
    words = ["".join(rng.choice("abc") for _ in range(rng.randrange(1, 5)))
             for _ in range(rng.randrange(0, 20))]
    pairs = [(w, rng.randrange(1, 6)) for w in words]
    prefix = "".join(rng.choice("abc") for _ in range(rng.randrange(0, 3)))
    return (pairs, prefix, rng.randrange(1, 5))


checked = stress(lambda c: Autocomplete(c[0]).complete(c[1], c[2]),
                 lambda c: complete_reference(c[0], c[1], c[2]),
                 gen_ac, n=4000, seed=RANDOM_SEED, label="autocomplete")
print("autocomplete: %s random (dictionary, prefix, k) cases agree with"
      % "{:,}".format(checked))
print("              sort-and-slice, including ties and absent prefixes.")

ac = Autocomplete([("car", 50), ("card", 30), ("care", 90), ("careful", 20),
                   ("cart", 40), ("cat", 70), ("dog", 60)])
print()
for p in ("car", "ca", "d", "z"):
    print("  complete(%-5r, k=3) -> %s" % (p, ac.complete(p, 3)))

autocomplete: 4,000 random (dictionary, prefix, k) cases agree with
              sort-and-slice, including ties and absent prefixes.

  complete('car', k=3) -> ['care', 'car', 'cart']
  complete('ca' , k=3) -> ['care', 'cat', 'car']
  complete('d'  , k=3) -> ['dog']
  complete('z'  , k=3) -> []


In [8]:
# ---------------------------------------------------------------------------
# Trie vs sorted array for prefix queries -- the comparison that is actually fair.
# ---------------------------------------------------------------------------
def prefix_bisect(sorted_words, prefix):
    """All words with the given prefix, via two binary searches."""
    lo = bisect.bisect_left(sorted_words, prefix)
    hi = bisect.bisect_left(sorted_words, prefix + "￿")
    return sorted_words[lo:hi]


rng = random.Random(RANDOM_SEED)
corpus = random_words(100_000, rng)
trie = Trie(corpus)
array = sorted(corpus)

checked = stress(lambda p: trie.with_prefix(p),
                 lambda p: prefix_bisect(array, p),
                 lambda r: "".join(r.choice("abcde") for _ in range(r.randrange(0, 4))),
                 n=2000, seed=RANDOM_SEED, label="prefix agreement")
print("The trie and the sorted array agree on %s random prefix queries."
      % "{:,}".format(checked))

prefixes = ["".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(3))
            for _ in range(500)]
start = time.perf_counter()
for p in prefixes:
    trie.with_prefix(p)
t_trie = (time.perf_counter() - start) / len(prefixes) * 1e6
start = time.perf_counter()
for p in prefixes:
    prefix_bisect(array, p)
t_array = (time.perf_counter() - start) / len(prefixes) * 1e6

print()
print("  %s words, 3-character prefixes, %d queries:" % ("{:,}".format(len(corpus)), len(prefixes)))
print("    trie.with_prefix   %8.1f us" % t_trie)
print("    sorted + bisect    %8.1f us" % t_array)
print()
print("  The sorted array wins on the query, because two binary searches and a")
print("  slice beat a subtree traversal that rebuilds every string from its path.")
print("  The trie's advantage is that it supports insert and delete in")
print("  O(len(word)); keeping the array sorted costs O(n) per insertion.")
print()
print("  So: STATIC dictionary -> sorted array. CHANGING dictionary -> trie.")
del trie, array

The trie and the sorted array agree on 2,000 random prefix queries.

  100,000 words, 3-character prefixes, 500 queries:
    trie.with_prefix       18.1 us
    sorted + bisect         2.4 us

  The sorted array wins on the query, because two binary searches and a
  slice beat a subtree traversal that rebuilds every string from its path.
  The trie's advantage is that it supports insert and delete in
  O(len(word)); keeping the array sorted costs O(n) per insertion.

  So: STATIC dictionary -> sorted array. CHANGING dictionary -> trie.


## 2.2 Word search on a grid — pruning with many patterns at once

**The problem.** Given a grid of letters and a dictionary, find every dictionary word formable by
walking to adjacent cells without reusing one.

**Why the obvious approach fails.** Searching for each word separately means a fresh DFS per word:
with $W$ words that is $W$ traversals, each exploring paths that mostly go nowhere.

**What the trie changes.** Walk the grid **once**, carrying a trie node alongside the path. At each
step, if the next letter has no edge from the current node, **no word in the entire dictionary
starts with this path** — prune immediately. One DFS answers for all $W$ words simultaneously.

That is the general pattern and it is worth naming: **a trie turns "does any of these many patterns
match?" into a single $O(1)$ check per step.** The same idea is Aho-Corasick (NB-02 Q11), which adds
failure links to do it for substring search.

The measurement below counts *cells visited* with and without the trie, because that is what the
pruning changes — timing would conflate it with Python overhead.

In [9]:
# ---------------------------------------------------------------------------
# 2.2 Word search: one DFS for the whole dictionary.
# ---------------------------------------------------------------------------
def find_words(grid, dictionary):
    """Every dictionary word formable by a self-avoiding walk on the grid."""
    trie = Trie(dictionary)
    rows, cols = len(grid), len(grid[0]) if grid else 0
    found, visits = set(), 0

    def dfs(r, c, node, text, seen):
        nonlocal visits
        ch = grid[r][c]
        child = node.children.get(ch)
        if child is None:                          # PRUNE: no word has this prefix
            return
        visits += 1
        text += ch
        if child.is_word:
            found.add(text)
        seen.add((r, c))
        for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in seen:
                dfs(nr, nc, child, text, seen)
        seen.discard((r, c))

    for r in range(rows):
        for c in range(cols):
            dfs(r, c, trie.root, "", set())
    return sorted(found), visits


def find_words_bruteforce(grid, dictionary):
    """One DFS per word: the version without a trie."""
    rows, cols = len(grid), len(grid[0]) if grid else 0
    found, visits = set(), 0

    def dfs(r, c, word, i, seen):
        nonlocal visits
        if grid[r][c] != word[i]:
            return False
        visits += 1
        if i == len(word) - 1:
            return True
        seen.add((r, c))
        for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in seen:
                if dfs(nr, nc, word, i + 1, seen):
                    seen.discard((r, c))
                    return True
        seen.discard((r, c))
        return False

    for word in dictionary:
        if not word:
            continue
        for r in range(rows):
            for c in range(cols):
                if dfs(r, c, word, 0, set()):
                    found.add(word)
                    break
            else:
                continue
            break
    return sorted(found), visits


def gen_grid(rng):
    rows = rng.randrange(1, 4)
    cols = rng.randrange(1, 4)
    grid = [[rng.choice("abc") for _ in range(cols)] for _ in range(rows)]
    words = ["".join(rng.choice("abc") for _ in range(rng.randrange(1, 5)))
             for _ in range(rng.randrange(1, 8))]
    return (grid, sorted(set(words)))


checked = stress(lambda c: find_words(c[0], c[1])[0],
                 lambda c: find_words_bruteforce(c[0], c[1])[0],
                 gen_grid, n=3000, seed=RANDOM_SEED, label="find_words")
print("find_words: %s random (grid, dictionary) pairs agree with the" % "{:,}".format(checked))
print("            one-DFS-per-word version.")

grid = [list("oaan"), list("etae"), list("ihkr"), list("iflv")]
words = ["oath", "pea", "eat", "rain", "oat", "hike", "tea"]
found, visits = find_words(grid, words)
print()
print("  grid:", " / ".join("".join(row) for row in grid))
print("  dictionary:", words)
print("  found:", found)

print()
print("Cells visited, with and without the trie:")
print("  %10s %16s %18s %12s" % ("words", "trie visits", "per-word visits", "ratio"))
rng = random.Random(RANDOM_SEED)
big_grid = [[rng.choice("abcde") for _ in range(6)] for _ in range(6)]
pool = sorted({"".join(rng.choice("abcde") for _ in range(rng.randrange(3, 6)))
               for _ in range(400)})
for count in (25, 100, 400):
    subset = pool[:count]
    _, v_trie = find_words(big_grid, subset)
    _, v_brute = find_words_bruteforce(big_grid, subset)
    print("  %10d %16s %18s %11.1fx"
          % (count, "{:,}".format(v_trie), "{:,}".format(v_brute), v_brute / max(v_trie, 1)))
print()
print("  The trie version visits about four times fewer cells at every")
print("  dictionary size. Note that BOTH grow: a larger dictionary means more")
print("  valid prefixes, so the trie prunes less often. What the trie buys is")
print("  a constant factor from sharing one grid walk -- not independence from")
print("  the dictionary size, which would be too strong a claim.")

find_words: 3,000 random (grid, dictionary) pairs agree with the


            one-DFS-per-word version.

  grid: oaan / etae / ihkr / iflv
  dictionary: ['oath', 'pea', 'eat', 'rain', 'oat', 'hike', 'tea']
  found: ['eat', 'oat', 'oath']

Cells visited, with and without the trie:
       words      trie visits    per-word visits        ratio
          25               44                189         4.3x
         100              242                983         4.1x
         400              752              3,544         4.7x

  The trie version visits about four times fewer cells at every
  dictionary size. Note that BOTH grow: a larger dictionary means more
  valid prefixes, so the trie prunes less often. What the trie buys is
  a constant factor from sharing one grid walk -- not independence from
  the dictionary size, which would be too strong a claim.


## 2.3 Longest common prefix — a walk down the single-child chain

**The problem.** The longest prefix shared by every string in a set.

**With a trie it is a definition, not an algorithm:** insert everything, then walk from the root
while the current node has **exactly one child and is not itself a word**. The moment the tree
branches, the strings have diverged; the moment a node is a word, that string has ended.

That second condition is the one people forget — `["ab", "abc"]` has LCP `"ab"`, and the node for
`ab` has one child but is a word, so the walk must stop there.

**Is a trie the right tool here?** Honestly, no — the direct answer (compare characters across all
strings, column by column) is $\Theta(\text{total input})$ with no structure at all, and a trie is
$\Theta(\text{total input})$ to *build* plus a walk. The trie version is worth seeing because it
makes the answer obvious rather than because it is faster, and because it generalises: with the trie
already built, the LCP of every *subset* sharing a prefix is free.

In [10]:
# ---------------------------------------------------------------------------
# 2.3 Longest common prefix, two ways.
# ---------------------------------------------------------------------------
def lcp_trie(words):
    if not words:
        return ""
    trie = Trie(words)
    node, out = trie.root, []
    while len(node.children) == 1 and not node.is_word:
        ch, child = next(iter(node.children.items()))
        out.append(ch)
        node = child
    return "".join(out)


def lcp_direct(words):
    if not words:
        return ""
    shortest = min(words, key=len)
    for i, ch in enumerate(shortest):
        for w in words:
            if w[i] != ch:
                return shortest[:i]
    return shortest


checked = stress(lcp_trie, lcp_direct,
                 lambda r: [("".join(r.choice("ab") for _ in range(r.randrange(0, 6))))
                            for _ in range(r.randrange(0, 8))],
                 n=4000, seed=RANDOM_SEED, label="lcp")
print("lcp: %s random word lists agree with the column-scan version," % "{:,}".format(checked))
print("     including the empty list, empty strings and single words.")

print()
for group in (["flower", "flow", "flight"], ["ab", "abc"], ["dog", "racecar"], []):
    print("  %-32s -> %r" % (group, lcp_trie(group)))
print()
print("  ['ab', 'abc'] is the case that catches people: the node for 'ab' has")
print("  exactly one child, so a walk that only checks branching runs past the")
print("  end of a word. The is_word test is what stops it.")

lcp: 4,000 random word lists agree with the column-scan version,
     including the empty list, empty strings and single words.

  ['flower', 'flow', 'flight']     -> 'fl'
  ['ab', 'abc']                    -> 'ab'
  ['dog', 'racecar']               -> ''
  []                               -> ''

  ['ab', 'abc'] is the case that catches people: the node for 'ab' has
  exactly one child, so a walk that only checks branching runs past the
  end of a word. The is_word test is what stops it.


## 2.4 Maximum XOR pair — a trie that holds no strings

**The problem.** Given $n$ integers, find the maximum value of $a \oplus b$ over all pairs.

Brute force is $\Theta(n^2)$. The trie solution is $\Theta(n \cdot 32)$, and it is here because it
makes a point the string examples cannot:

> **A trie is not a string structure.** It is a structure over any key that can be read as a
> **sequence of symbols**. Here the symbols are **bits**, the alphabet is $\{0, 1\}$, and every key
> has exactly 32 of them.

**The algorithm.** Insert every number as a 32-bit path, most significant bit first. Then for each
number, walk down **greedily taking the opposite bit** whenever that child exists — because a
differing bit at position $i$ contributes $2^i$, and the highest position dominates every lower one
combined. If the opposite child is missing, take the same bit and continue.

The greedy choice is provably optimal precisely because of that dominance: $2^i > \sum_{j<i} 2^j$.
This is the same reason binary search on a sorted array works, applied one bit at a time.

**Where this really lives:** IP routing tables are exactly this structure, matching the
longest prefix of a destination address against a trie of network prefixes. Linux's routing table
is a compressed bitwise trie — a radix tree (§1.5) over bits.

In [11]:
# ---------------------------------------------------------------------------
# 2.4 A bitwise trie: maximum XOR of any pair.
# ---------------------------------------------------------------------------
BITS = 32


class BitTrie:
    """A trie over the bits of an integer, most significant first."""

    def __init__(self, numbers=()):
        self.root = [None, None]                 # children[0], children[1]
        for x in numbers:
            self.insert(x)

    def insert(self, x):
        node = self.root
        for i in range(BITS - 1, -1, -1):
            b = (x >> i) & 1
            if node[b] is None:
                node[b] = [None, None]
            node = node[b]

    def best_xor_with(self, x):
        """The largest x ^ y over the stored y."""
        node, total = self.root, 0
        for i in range(BITS - 1, -1, -1):
            b = (x >> i) & 1
            want = 1 - b                          # a differing bit is worth 2^i
            if node[want] is not None:
                total |= 1 << i
                node = node[want]
            else:
                node = node[b]                    # forced to agree at this bit
        return total


def max_xor_pair(numbers):
    if len(numbers) < 2:
        return 0
    trie = BitTrie()
    best = 0
    trie.insert(numbers[0])
    for x in numbers[1:]:
        best = max(best, trie.best_xor_with(x))
        trie.insert(x)
    return best


def max_xor_bruteforce(numbers):
    best = 0
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            best = max(best, numbers[i] ^ numbers[j])
    return best


checked = stress(max_xor_pair, max_xor_bruteforce,
                 lambda r: [r.randrange(0, 1 << 16) for _ in range(r.randrange(0, 30))],
                 n=4000, seed=RANDOM_SEED, label="max_xor_pair")
print("max_xor_pair: %s random integer lists agree with the O(n^2) reference,"
      % "{:,}".format(checked))
print("              including empty lists, single elements and duplicates.")

print()
demo = [3, 10, 5, 25, 2, 8]
print("  numbers:", demo)
print("  max XOR pair:", max_xor_pair(demo), " (25 ^ 5 = %d)" % (25 ^ 5))

print()
print("Growth, against the quadratic reference:")
def make_numbers(n):
    r = random.Random(n)
    return [r.randrange(0, 1 << 31) for _ in range(n)]

print("  brute force:")
growth_table(measure_growth(max_xor_bruteforce, [500, 1_000, 2_000], setup=make_numbers,
                            repeats=3), claim="O(n^2)")
print()
print("  bitwise trie:")
growth_table(measure_growth(max_xor_pair, [20_000, 40_000, 80_000], setup=make_numbers,
                            repeats=3), claim="O(n)")

max_xor_pair: 4,000 random integer lists agree with the O(n^2) reference,
              including empty lists, single elements and duplicates.

  numbers: [3, 10, 5, 25, 2, 8]
  max XOR pair: 28  (25 ^ 5 = 28)

Growth, against the quadratic reference:
  brute force:


         n        seconds      ratio
------------------------------------
       500       0.016351          -
     1,000       0.065748       4.02
     2,000       0.267752       4.07

best fit: O(n^2) (relative error 0.010); next: O(n log n) (0.652)
claimed O(n^2) -> measurement MATCHES the claim

  bitwise trie:


         n        seconds      ratio
------------------------------------
    20,000       0.393661          -
    40,000       0.763504       1.94
    80,000       1.540898       2.02

best fit: O(n) (relative error 0.013); next: O(n log n) (0.063)
claimed O(n) -> measurement MATCHES the claim


[('O(n)', 0.012852806869128004),
 ('O(n log n)', 0.06284550633529307),
 ('O(log n)', 0.670311503114883),
 ('O(1)', 0.7863795610507016),
 ('O(2^n)', 0.7863795610507016),
 ('O(n^2)', 0.8276911774500738),
 ('O(n^3)', 3.5691232626800034)]

***
# Part 3 - The signature difficulty: memory

Every other structure in this series has been cheap to store. An array is its elements; a heap is
its elements; a balanced tree is its elements plus two pointers each. **A trie is not.** It stores
one node per *distinct prefix*, and each node carries a child map.

So the question is not "does a trie use more memory" — obviously it does — but **how much more, and
what determines it.** The answer is **prefix sharing**, and the two corpora below are chosen to sit
at opposite ends of it:

- **low sharing** — random strings over 26 letters, so almost every word forks near the root;
- **high sharing** — words built from a small set of stems with suffixes, which is roughly what a
  natural-language dictionary looks like.

Same word count, same code, and the memory differs by a factor that decides whether the structure is
usable.

In [12]:
# ---------------------------------------------------------------------------
# 3.1 The corpora, and what a trie costs on each.
# ---------------------------------------------------------------------------
STEMS = ["compute", "comput", "connect", "consider", "construct", "contain", "content",
         "process", "product", "program", "project", "protect", "provide",
         "system", "syntax", "synth", "search", "second", "section", "secure",
         "trans", "transfer", "transform", "translate", "transport"]
SUFFIXES = ["", "s", "ed", "ing", "er", "ers", "ion", "ions", "able", "ability",
            "ment", "ments", "ive", "ively", "or", "ors", "al", "ally"]


def shared_prefix_words(count, rng):
    """Words built from a small set of stems: heavy prefix sharing."""
    out = set()
    while len(out) < count:
        word = rng.choice(STEMS) + rng.choice(SUFFIXES)
        if rng.random() < 0.5:
            word += rng.choice(SUFFIXES)
        out.add(word)
        if len(out) < count:
            out.add(word + "".join(rng.choice("abcdefghijklmnopqrstuvwxyz")
                                   for _ in range(rng.randint(1, 3))))
    return sorted(out)[:count]


def trie_bytes(root):
    """Deep size: every node plus its children dict. (NB-01 section 1.4's lesson --
    sys.getsizeof alone reports the struct and omits everything it points at.)"""
    total, stack = 0, [root]
    while stack:
        node = stack.pop()
        total += sys.getsizeof(node) + sys.getsizeof(node.children)
        stack.extend(node.children.values())
    return total


def radix_bytes(root):
    total, stack = 0, [root]
    while stack:
        node = stack.pop()
        total += sys.getsizeof(node) + sys.getsizeof(node.children)
        for label, child in node.children.values():
            total += sys.getsizeof(label)
            stack.append(child)
    return total


def set_bytes(words):
    return sys.getsizeof(set(words)) + sum(sys.getsizeof(w) for w in words)


N = 20_000
corpora = [("low sharing", random_words(N, random.Random(11))),
           ("high sharing", shared_prefix_words(N, random.Random(11)))]

print("%s words each. 'nodes/word' is the whole story." % "{:,}".format(N))
print()
print("  %-14s %12s %12s %14s %14s %10s"
      % ("corpus", "avg length", "trie nodes", "nodes/word", "trie bytes", "vs set"))
print("  " + "-" * 80)
for label, words in corpora:
    trie = Trie(words)
    tb, sb = trie_bytes(trie.root), set_bytes(words)
    print("  %-14s %12.1f %12s %14.2f %14s %9.1fx"
          % (label, statistics.mean(len(w) for w in words),
             "{:,}".format(trie.node_count()), trie.node_count() / len(words),
             "{:,}".format(tb), tb / sb))
    del trie
print()
print("  set bytes, for reference: %s and %s"
      % ("{:,}".format(set_bytes(corpora[0][1])), "{:,}".format(set_bytes(corpora[1][1]))))
print()
print("Prefix sharing decides everything. With random strings almost every word")
print("forks near the root, so the trie needs a node for nearly every character")
print("of every word. With shared stems it needs a fraction of that.")

20,000 words each. 'nodes/word' is the whole story.

  corpus           avg length   trie nodes     nodes/word     trie bytes     vs set
  --------------------------------------------------------------------------------


  low sharing             8.5      122,294           6.11     26,196,280       8.5x
  high sharing           13.7       38,913           1.95      7,512,856       2.4x

  set bytes, for reference: 3,087,408 and 3,192,300

Prefix sharing decides everything. With random strings almost every word
forks near the root, so the trie needs a node for nearly every character
of every word. With shared stems it needs a fraction of that.


In [13]:
# ---------------------------------------------------------------------------
# 3.2 Now compress it, and measure again.
# ---------------------------------------------------------------------------
print("The same corpora, as radix trees:")
print()
print("  %-14s %12s %12s %10s %14s %14s %10s"
      % ("corpus", "trie nodes", "radix nodes", "saving", "trie bytes", "radix bytes", "vs set"))
print("  " + "-" * 92)
for label, words in corpora:
    trie, radix = Trie(words), RadixTree(words)
    assert trie.words() == radix.words(), "the two structures disagree"
    tb, rb, sb = trie_bytes(trie.root), radix_bytes(radix.root), set_bytes(words)
    print("  %-14s %12s %12s %9.2fx %14s %14s %9.1fx"
          % (label, "{:,}".format(trie.node_count()), "{:,}".format(radix.node_count()),
             trie.node_count() / radix.node_count(),
             "{:,}".format(tb), "{:,}".format(rb), rb / sb))
    del trie, radix

The same corpora, as radix trees:

  corpus           trie nodes  radix nodes     saving     trie bytes    radix bytes     vs set
  --------------------------------------------------------------------------------------------


  low sharing         122,294       26,528      4.61x     26,196,280      5,188,468       1.7x


  high sharing         38,913       21,992      1.77x      7,512,856      4,527,727       1.4x


**Compression works, and it is not enough.**

On the low-sharing corpus the radix tree removes about **four fifths of the nodes** and cuts memory
by a similar factor — a large, real win, and exactly what collapsing single-child chains should do
when almost every chain is long. On the high-sharing corpus the saving is smaller, because there
were fewer single-child chains to collapse: the words already shared their prefixes, which is what
the plain trie was good at.

**And in both cases the radix tree is still larger than a plain `set`.** That is the honest verdict
of this part:

> Even after doing everything right, a trie costs more memory than the flat structure it replaces.
> You are not buying lookup speed — §1.3 showed the hash set wins that comfortably. You are buying
> **prefix queries**, and the memory is the price.

**When the price is worth paying:**

- **The query is genuinely a prefix.** Autocomplete, routing tables, spell-check candidates,
  filesystem path resolution. Nothing else answers these.
- **The dictionary changes.** §2.1 measured a sorted array *beating* the trie on prefix queries — so
  for a static dictionary, use the array. The trie earns its memory when insert and delete must be
  $O(|\text{key}|)$ rather than $O(n)$.
- **Many patterns, one pass.** §2.2's grid search, and Aho-Corasick.
- **The alphabet is tiny.** §2.4's bitwise trie has two children per node, so the per-node overhead
  is minimal and the structure is genuinely compact — which is why IP routing uses one.

**When it is not:**

- **Exact lookup only.** Use a hash set. It is faster and smaller, measured both ways.
- **Long keys with little sharing.** The worst case for a trie: one node per character with nothing
  amortised. §3.1's low-sharing corpus is this, and it costs several times a `set`.
- **Large alphabets with array nodes.** §1.4 measured 3.8% slot utilisation for 26 letters; for
  Unicode it is not expressible at all.

**One more option worth knowing**, because it dominates tries for static dictionaries: a
**DAWG** (directed acyclic word graph) merges identical *suffixes* as well as prefixes, turning the
tree into a DAG. For a natural-language word list it can be an order of magnitude smaller than a
trie and still answer prefix queries — at the cost of being unable to accept insertions at all.

***
# Part 4 - Tough questions

***

### Q1. What is a trie, and how does it differ from a hash table and a search tree?

<details><summary>Answer</summary>

A tree whose **edges are symbols** and whose paths spell out the keys. A node stores no key — its
identity is the path taken to reach it — plus a flag saying whether that path is itself a stored key.

| | Locates a key by | Cost | Extra capability |
|---|---|---|---|
| hash table (NB-03) | hashing the **whole key** | $O(1)$ expected | none |
| balanced tree (NB-08) | **comparing whole keys** | $O(\log n)$ comparisons | order, ranges |
| **trie** | **walking the key's symbols** | $O(\lvert\text{key}\rvert)$ | **prefixes** |

**A trie never compares two keys and never hashes one.** That gives two properties:

1. **The cost does not depend on $n$** — §1.3 measures the operation count as exactly flat (8.5 hops)
   while the collection grows 200×.
2. **Everything sharing a prefix lives under one node**, so prefix queries are a walk plus a subtree
   traversal. A hash table cannot do this at any price, because hashing destroys the relationship
   between `pre` and `prefix` by design.

**The `is_word` flag is not an implementation detail.** Without it you cannot tell "`car` is a stored
key" from "`car` is only a prefix of `cart`", and that distinction *is* the difference between
`search` and `starts_with`.

</details>

***

### Q2. Is a trie lookup really independent of the number of keys?

<details><summary>Answer</summary>

**In operations, yes — exactly. In time, no.** §1.3 measures both because they disagree.

The hops column is flat at ~8.5 as the key count grows from 1,000 to 200,000: one edge per symbol,
and the number of symbols does not change. The claim is true and counting is what shows it.

The wall clock over the same range **grows about 1.7×**, and the reason is the one this series keeps
returning to: **a trie is a pointer structure with no locality.** Every hop is a dict lookup in a
separately allocated node, and a 200,000-word trie has over a million nodes scattered across the
heap. At a thousand words it fits in cache; at 200,000 it does not. The *number* of hops is
unchanged; the *cost of a hop* is not.

This is NB-09 §2.4's observation arriving from the other side. There, a heap had a perfect flat
layout and still lost to quicksort because the *access pattern* strided badly. Here the access
pattern is fine and there is no layout at all.

**So state the property precisely:** a trie's operation count is genuinely independent of $n$; its
running time is not, because memory is not flat. The growth is a bounded constant-factor effect, not
a complexity change — and note the `bisect` column grows too.

**And the comparison that matters:** for *exact* lookup a hash set beats the trie comfortably at
every size, because one hash of an 8-character string is cheaper than eight dict lookups. **A trie
is not a faster way to answer the question a hash table answers.**

</details>

***

### Q3. Walk through insert, search and delete.

<details><summary>Answer</summary>

**Insert:** walk the symbols, creating missing nodes, then set `is_word`. $O(|\text{key}|)$.

**Search:** walk; present iff you arrive *and* the flag is set. Arriving is not enough.

**Delete** is the only interesting one: clear the flag, then **prune** back up the path, removing
every node that now has no flag and no children. Stop at the first node that is still useful.

**Why pruning is not optional:** skip it and the trie keeps a node for every key ever inserted, for
the life of the process. Nothing fails — searches still return the right answers — and the structure
leaks. §3 is about memory, so this is the one bug in the notebook that costs exactly the resource the
whole part is about, and it is invisible to a correctness test.

**Hence the invariant**, checked after every operation in §1.2: *every node leads to at least one
word.* That is a structural claim, not a behavioural one, and it is the only thing that catches a
missing prune — NB-08 §3's argument in a new setting.

**And the case the randomised test found on its own:** `starts_with("")` on an **empty** trie. The
root always exists, so walking succeeds and a naive implementation answers `True`. The fix is to
require a word at or below the node reached.

</details>

***

### Q4. Array children or map children?

<details><summary>Answer</summary>

| | Array (`Node[26]`) | Map (`dict` / `HashMap`) |
|---|---|---|
| lookup | one subtraction, one index | hash the symbol, probe |
| memory | one slot per **alphabet symbol** per node | one entry per **actual** child |
| alphabet | must be small and fixed | anything, including Unicode |

§1.4 measures both in Java on 100,000 words: the array form is meaningfully faster — no boxing, no
hashing, just arithmetic — and the memory line is the one to remember:

> 557,021 nodes × 26 slots = 14.5 million references, of which **3.8% are used**. The average node
> has **1.00 children**.

That is the array's whole problem. Most trie nodes are deep, where the structure has stopped
branching, so 25 of every 26 slots are `null`. The array is fast because it does no work to find a
child, and wasteful for exactly the same reason.

**Choose by alphabet:** tiny and fixed (DNA, bits, digits) → array, and the slots actually get used.
Lowercase ASCII → a real trade. Unicode → a map, necessarily, since the array is not expressible.
Sparse deep nodes → a map, or better, a **radix tree** (§1.5), which removes those nodes entirely
rather than choosing a cheaper representation for them.

**And the caution from NB-02 §1.4:** indexing by "character" assumes a character is one code unit.
Beyond ASCII it is not — a trie over Python `str` walks *code points*, so combining marks take
several edges and NFC/NFD normalisation changes the tree's shape. Normalise at the boundary or the
same visible word takes two different paths.

</details>

***

### Q5. What is a radix tree, and what does it buy?

<details><summary>Answer</summary>

A trie in which every chain of **single-child** nodes is collapsed into one edge carrying the whole
substring. A node then exists only where keys genuinely branch.

**Why it matters:** in a plain trie, a path like `d → o → g` is three nodes carrying no branching
information at all, and over realistic keys those nodes are the overwhelming majority.

§3 measures the saving on 20,000 words:

| corpus | trie nodes | radix nodes | saving | trie bytes | radix bytes |
|---|---|---|---|---|---|
| low sharing | 122,294 | 26,528 | **4.61×** | 26.2 MB | **5.2 MB** |
| high sharing | 38,913 | 21,992 | 1.77× | 7.5 MB | 4.5 MB |

The saving is largest exactly where the plain trie is worst — random strings, long single-child
chains. On already-well-shared words there was less to collapse.

**What it costs is the insert.** Arriving at an edge labelled `care` while inserting `cart` means
**splitting** the edge: a new node at the common prefix `car`, with `e` and `t` below. That split,
and its sub-case where the split point is itself the new word, is where implementations break — which
is why §1.5's invariant checks every edge label is non-empty and keyed by its first symbol.

**And the honest verdict:** even compressed, both corpora remain **larger than a plain `set`**. You
are not buying lookup speed. You are buying prefix queries, and the memory is the price.

**Where it is used:** Linux's IP routing table, Java's `ConcurrentRadixTree` implementations, and
most production "trie" libraries — because the plain version's memory is rarely acceptable.

</details>

***

### Q6. Why is memory the trie's signature difficulty?

<details><summary>Answer</summary>

Because it stores **one node per distinct prefix**, each with a child map — and unlike every other
structure in this series, that is not proportional to the data.

§3 measures 20,000 words against a plain `set`:

- **low sharing** (random strings): 6.11 nodes per word, **8.5× the memory of a `set`**;
- **high sharing** (shared stems): 1.95 nodes per word, **2.4×**.

**Prefix sharing is the entire economics**, and it is a property of your data, not of your code. The
same implementation is comfortably usable on one corpus and wasteful on the other.

**Compression helps and does not close the gap** (Q5): a radix tree gets low-sharing down to 1.7×
and high-sharing to 1.4×, still above the `set`.

**So the decision rule:**

- **Exact lookup only?** Use a hash set — faster *and* smaller, measured both ways.
- **Long keys with little prefix sharing?** The trie's worst case; one node per character with
  nothing amortised.
- **Static dictionary?** §2.1 measured a sorted array **beating** the trie on prefix queries. Use
  the array.
- **Prefix queries on a changing dictionary?** Now the trie earns it — insert and delete are
  $O(|\text{key}|)$ where the array is $O(n)$.
- **Tiny alphabet?** §2.4's bitwise trie has two children per node, so the overhead is minimal.

**And one more option**: a **DAWG** merges identical *suffixes* as well as prefixes, turning the
tree into a DAG. For a natural-language word list it can be an order of magnitude smaller than a
trie and still answer prefix queries — at the cost of accepting no insertions at all.

</details>

***

### Q7. How would you build autocomplete?

<details><summary>Answer</summary>

**The structure:** a trie, with a frequency stored at each word node. Walk to the prefix, traverse
its subtree, and keep the best $k$.

**The ranking is what makes it autocomplete** rather than a prefix dump, and §2.1 got it wrong
first: I encoded "most frequent, then alphabetical" as a hand-built comparable tuple, and a
randomised test found the failure immediately — `aca` and `acaa` with equal frequency, where the
encoding orders by length rather than alphabetically once one word is a prefix of another.

**The fix is to use a key function rather than an encoding:** `heapq.nsmallest(k, matches,
key=lambda w: (-freq[w], w))`. It keeps the size-$k$ heap of NB-09 §2.1 internally and gets the
compound ordering right by construction. **Do not hand-encode a multi-field ordering into a single
comparable value** — that is the same class of error as NB-07 §1.4's inconsistent comparator.

**And the honest alternative, from §2.1:** for a **static** dictionary, a sorted array with two
binary searches beats the trie on the query, and by a wide margin — the trie has to rebuild each
string from its path while the array just slices. Use the trie when the dictionary changes.

**Scaling it up in practice:** precompute the top-$k$ at each node so a query is a walk with no
traversal at all; shard by first letter; and for very large systems, an FST (finite state
transducer, as in Lucene) which is a DAWG with outputs and is what real search engines use.

</details>

***

### Q8. Where do tries appear that are not about words?

<details><summary>Answer</summary>

A trie works on **any key readable as a sequence of symbols**, which is much broader than strings.

- **IP routing tables.** The canonical non-string use: a bitwise trie over address bits, doing
  *longest-prefix match* to pick a route. Linux uses a compressed version (LC-trie), and this is
  what the phrase "prefix" in "IP prefix" literally means.
- **§2.4's maximum XOR pair.** A trie over 32 bits with a two-symbol alphabet, turning an
  $\Theta(n^2)$ pairwise search into $\Theta(32n)$ — measured and verified against brute force.
- **Filesystem paths.** A path *is* a sequence of components, and resolution is a trie walk.
- **Suffix trees and suffix arrays** (NB-02's territory): a trie of every suffix of a string,
  compressed, which answers substring queries in $O(m)$.
- **Aho-Corasick** (NB-02 Q11): a trie of many patterns plus failure links, matching all of them in
  one pass — §2.2's grid search generalised.
- **Persistent data structures.** Clojure's vectors and maps, and Scala's, are 32-way tries over the
  *bits of the index*, which is how they get effectively-constant-time access with structural
  sharing.

**The unifying idea worth taking away:** if your keys have internal structure you can walk, a trie
lets you index by that structure instead of flattening it into a hash or a comparison. That is a
different *kind* of indexing, not a faster version of the same one.

</details>

***

### Q9. Compare a trie with a sorted array for prefix queries.

<details><summary>Answer</summary>

This is the comparison that actually matters, and §2.1 measures it — because the usual comparison
(trie versus hash table) is unfair in the trie's favour, since a hash table cannot do prefix queries
at all.

**A sorted array can.** All strings with prefix $p$ form a **contiguous block**, found by two binary
searches: `bisect_left(p)` and `bisect_left(p + '￿')`. $O(\log n + k)$.

**Measured on 100,000 words with 3-character prefixes, the sorted array wins by roughly 8×.** Two
binary searches and a slice beat walking a subtree and rebuilding every result string from its path.

**So when is the trie right?**

- **When the dictionary changes.** Insert and delete are $O(|\text{key}|)$ in a trie and $O(n)$ in a
  sorted array, which must shift elements to stay sorted. This is the trie's actual advantage.
- **When you need more than prefix ranges** — §2.2's simultaneous multi-pattern matching, wildcard
  search, or edit-distance search, none of which a sorted array supports.
- **When you want the top-$k$ per prefix precomputed** at internal nodes.

**And the memory, from §3:** the array stores the strings and nothing else; the trie stores several
times that. So for a **static** dictionary the array is smaller *and* faster, which is a clean
verdict and not the one most treatments give.

</details>

***

### Q10. What is a trie bad at?

<details><summary>Answer</summary>

- **Memory** (§3): 8.5× a `set` with low prefix sharing, 2.4× with high sharing, and still 1.4–1.7×
  after radix compression. This is the signature difficulty.
- **Exact lookup**: a hash set is faster at every size measured (§1.3). Using a trie because it is
  "$O(\text{length})$" misreads what the constant is.
- **Locality**: no contiguity, one allocation per node, a pointer hop per symbol. §1.3 shows the
  wall clock growing with $n$ even though the operation count does not.
- **Non-prefix queries**: "words *ending* in `-ing`" needs a second trie over reversed words;
  "words *containing* `ss`" needs a suffix structure. A trie indexes prefixes and only prefixes.
- **Large alphabets**: array nodes become impossible and map nodes become the dominant cost. For
  Unicode this is decisive.
- **Long keys with no sharing**: the pathological case — one node per character, nothing amortised,
  and the memory of §3's low-sharing corpus.

**And a subtler one:** a trie is **ordered by symbol**, which is not necessarily the order you want.
Its traversal gives lexicographic order by code point, so `Z` precedes `a` and accented characters
sort after all unaccented ones. If you need locale-aware collation, a trie over raw code points is
the wrong index — the same class of problem as NB-02 §1.4's normalisation warning.

</details>

***

### Q11. How do you handle keys that are not simple ASCII strings?

<details><summary>Answer</summary>

**Decide what a "symbol" is, explicitly, and normalise at the boundary.** NB-02 §1.4 established
that "one character" has at least four meanings; a trie forces you to pick one, because the choice
*is* the edge alphabet.

- **Bytes** (alphabet 256): works for anything, makes the trie deeper, and is what a routing table
  or a binary-key store uses. UTF-8's self-synchronising design means byte-level prefixes still
  correspond to string prefixes, which is a genuinely useful property.
- **Code points** (Python `str` iteration): the natural choice, alphabet over a million, so map
  children are mandatory. Combining marks take their own edge, so `é` may be one edge or two
  depending on normalisation — **normalise to NFC on insert and on query**, or the same visible
  word takes two different paths and lookups silently miss.
- **Grapheme clusters**: what a user calls a character. Needs a segmentation library, and it is the
  right unit if the trie is user-facing.

**The failure mode is silent**, which is what makes it worth planning: a trie built from NFC data
and queried with NFD input simply reports the word absent. No exception, no warning — exactly the
shape of bug NB-02 §1.4 measured on string equality.

**Case-insensitivity** is the same decision: casefold on the way in *and* on the way out, or store
both. And note that case folding is locale-dependent (Turkish dotless ı), so "just call `.lower()`"
is another place where the honest answer is "decide, then write it down".

</details>

***

### Q12. Trie, hash table, or balanced tree — how do you choose?

<details><summary>Answer</summary>

Three notebooks, three indexing strategies, and the choice comes down to **what question you need to
ask**:

| Question | Structure | Why |
|---|---|---|
| "is this exact key present?" | **hash table** (NB-03) | $O(1)$ expected, smallest memory |
| "what keys are between $a$ and $b$?" | **balanced tree** (NB-08) | ordering, ranges, successor |
| "what keys start with $p$?" | **trie** | prefixes are structural, not comparable |
| "what keys start with $p$, static data" | **sorted array** | §2.1 measured it beating the trie |
| "do any of these many patterns match?" | **trie** (§2.2) or Aho-Corasick | one pass for all patterns |

**The decision is about capability, not speed.** A trie is not a faster hash table — §1.3 measures it
losing on exact lookup at every size, and §3 measures it costing several times the memory. It answers
a question the others cannot, and you pay for that capability.

**The order to think in:**

1. Do I need prefix or multi-pattern queries at all? If no → hash table, or a tree if you need order.
2. Does the dictionary change? If no → sorted array with binary search, which is smaller and faster.
3. Is prefix sharing high? If no, budget for §3's memory or use a radix tree from the start.
4. Is the alphabet small? If yes, array nodes; if it is Unicode, map nodes and normalisation.

**And the meta-point this notebook adds to the series:** the same word "lookup" has covered three
genuinely different mechanisms — hashing the key, comparing keys, and walking the key. They are not
interchangeable implementations of one idea, and knowing which question you are asking is what picks
between them.

</details>

***

## Coding challenges

### Challenge 1 — a DAWG, and measure it against the radix tree

§3 named the structure that beats both. Build it.

1. Build a trie, then **merge identical subtrees** so that suffixes are shared as well as prefixes —
   turning the tree into a DAG. Hash each subtree by its canonical form (children plus `is_word`)
   and reuse nodes with the same hash.
2. Verify it accepts exactly the same word set as the trie, over thousands of randomised
   dictionaries.
3. Measure nodes and bytes against the trie and the radix tree on §3's two corpora, and against a
   plain `set`. It should be the first structure in the notebook that *beats* the set.
4. Then explain why it cannot accept insertions, and what that costs you in practice.

### Challenge 2 — approximate matching in a trie

§3 says a trie only answers prefix queries. Extend it to edit distance.

1. Implement "all words within edit distance $d$ of a query", walking the trie while carrying a row
   of the dynamic-programming table (NB-19 previews the DP). Prune a subtree when the whole row
   exceeds $d$.
2. Verify against brute force — compute the edit distance to every word — on small dictionaries.
3. Measure the pruning: count nodes visited against the trie's total size, for $d = 1, 2, 3$.
4. This is how a spell-checker actually works, and it is the strongest argument in the notebook for
   a trie over a sorted array. Say why the array cannot do it.

### Challenge 3 — the routing table

§2.4 and Q8 claim tries are not a string structure. Build the canonical non-string one.

1. Implement **longest-prefix match** over IPv4: insert network prefixes like `10.0.0.0/8` into a
   bitwise trie, then look up an address and return the *most specific* matching prefix.
2. Verify against a linear scan that checks every prefix and takes the longest match.
3. Measure lookups against the linear scan as the table grows to a realistic size (tens of thousands
   of routes).
4. Then compress it into a radix tree and measure again — this is essentially what Linux does, and
   the memory saving is the reason.

***
# Part 5 - Practice

| # | Exercise | The technique | Difficulty |
|---|---|---|---|
| 1 | Implement a trie | Insert, search, `starts_with`, and pruning delete | ★★☆☆☆ |
| 2 | Replace words with roots | Shortest-prefix match | ★★☆☆☆ |
| 3 | Design an autocomplete system | Trie + ranking | ★★★☆☆ |
| 4 | Word search II | Trie-pruned DFS on a grid | ★★★★☆ |
| 5 | Maximum XOR of two numbers | A bitwise trie | ★★★★☆ |
| 6 | Wildcard search (`.` matches any) | Branching walk | ★★★☆☆ |
| 7 | Longest word built one letter at a time | Trie plus a validity check | ★★★☆☆ |
| 8 | Palindrome pairs | Reversed trie plus palindrome checks | ★★★★★ |

***

### 1. Implement a trie

- **Brief:** `insert`, `search`, `starts_with`, and `delete` **with pruning**.
- **Good result:** differentially tested against a `set` over thousands of randomised operation
  sequences, with the no-dead-subtrees invariant asserted after every step (§1.2's harness).
- **The trap:** two of them. `search` must check the `is_word` flag, not just that the walk arrived.
  And `starts_with("")` on an **empty** trie must be `False` — the root always exists, so walking
  succeeding proves nothing. A randomised test finds the second one immediately; a hand-written test
  almost never does.

### 2. Replace words with roots

Given a dictionary of roots and a sentence, replace each word with the **shortest** root that is a
prefix of it.

- **Brief:** walk each word through the trie and stop at the **first** `is_word` node.
- **Good result:** $O(\text{total input})$, verified against a brute-force prefix check.
- **The trap:** "shortest" means stopping early — the walk must not continue to the longest match.
  Then do the variant asking for the *longest* match and note it is the routing-table rule (Q8).

### 3. Design an autocomplete system

- **Brief:** §2.1 — a trie with frequencies, returning the top $k$ by frequency then alphabetically.
- **Good result:** verified against sort-and-slice over randomised dictionaries including ties.
- **The trap:** the compound ordering. §2.1's first version hand-encoded "frequency then alphabet"
  into a comparable tuple and got it wrong whenever one candidate was a prefix of another. Use a
  **key function** (`heapq.nsmallest(k, matches, key=...)`), not an encoding.

### 4. Word search II

- **Brief:** §2.2 — one DFS over the grid carrying a trie node, pruning when a letter has no edge.
- **Good result:** verified against one-DFS-per-word, with the pruning measured in cells visited.
- **The trap:** two. Remove found words from the trie (or deduplicate the results) or you will report
  the same word many times. And prune trie nodes as they are exhausted, which keeps the later
  searches fast — a nice case where §1.2's delete-pruning pays off inside an algorithm.

### 5. Maximum XOR of two numbers

- **Brief:** §2.4 — a bitwise trie, walking greedily toward the opposite bit.
- **Good result:** $\Theta(32n)$, verified against the $\Theta(n^2)$ reference, with the growth
  measured.
- **The trap:** insert most-significant bit **first**. The greedy choice is only correct because a
  higher bit outweighs all lower bits combined ($2^i > \sum_{j<i} 2^j$) — build the trie
  least-significant-first and the greed is unjustified and wrong.

### 6. Wildcard search

Support `.` matching any single character.

- **Brief:** the walk becomes a branching search: on `.`, recurse into **every** child.
- **Good result:** verified against a regex or a linear scan over the dictionary.
- **The trap:** the complexity. A query of all dots visits the entire trie, so this is $O(n)$ in the
  worst case, not $O(|\text{query}|)$ — say so rather than claiming the trie's usual bound. Leading
  dots are the expensive case; trailing dots are cheap.

### 7. Longest word built one letter at a time

The longest dictionary word such that every prefix of it is also in the dictionary.

- **Brief:** insert everything, then DFS from the root descending **only** into children whose node
  is `is_word`. The deepest such path is the answer.
- **Good result:** $O(\text{total input})$, ties broken alphabetically, verified against a brute-force
  check of every word's prefixes.
- **The trap:** the tie-break, and the empty string — decide whether it counts as a prefix that must
  be present.

### 8. Palindrome pairs

All pairs $(i, j)$ where `words[i] + words[j]` is a palindrome.

- **Brief:** insert the words **reversed** into a trie. For each word, walk it through the trie; at
  each point where a stored reversed word ends, check whether the *remainder* is a palindrome.
- **Good result:** $O(n k^2)$ for $n$ words of length $k$, verified against the $O(n^2 k)$ brute
  force.
- **The trap:** the case analysis — one word longer, the other longer, equal lengths, empty strings,
  and not pairing a word with itself. This is the hardest problem in the notebook and it is
  essentially all edge cases, so build the brute force first and let it find them.

***
# Part 6 - Reading

## Start here

**1. [Fredkin, *Trie Memory*](https://dl.acm.org/doi/10.1145/367390.367400)** —
CACM 1960.
> The original, and the source of the name — Fredkin took it from *retrieval* and it has been
> mispronounced ever since. Worth reading for how directly it states the property §1.3 measures:
> access time depends on the key, not on the number of keys stored.

**2. [Morrison, *PATRICIA*](https://dl.acm.org/doi/10.1145/321479.321481)** —
JACM 1968.
> "Practical Algorithm To Retrieve Information Coded In Alphanumeric." The radix tree of §1.5,
> introduced for exactly the reason §3 measures: the plain trie's memory was unaffordable. Read it
> as a memory-optimisation paper, which is what it is.

**3. [Askitis & Sinha, *HAT-trie*](https://crpit.scem.westernsydney.edu.au/abstracts/CRPITV62Askitis.html)** —
2007. **Free.**
> A modern, cache-conscious answer to §1.3's wall-clock problem: a trie whose lower levels are
> replaced by small hash tables, so the pointer-chasing stops before it costs you. The best single
> reference for "why is my trie slow when its operation count is flat", and the measurements are
> the same shape as this notebook's.

## The source behind each section

| Section | Where it comes from | Free? |
|---|---|---|
| 1.1–1.2 — the trie | **Fredkin**, CACM 1960; **Knuth, TAOCP vol. 3 §6.3** — "Digital Searching" | 🔍 |
| 1.3 — the independence claim | **Knuth §6.3**; the cache caveat is measured here | 🔍 |
| 1.4 — node representation | **Bentley & Sedgewick**, *Fast algorithms for sorting and searching strings*, SODA 1997 — ternary search tries, the third option | 🔍 |
| 1.5 — **radix trees** | **Morrison**, JACM 1968 | 🔍 |
| 2.2 — multi-pattern search | **Aho & Corasick**, CACM 1975 | 🔍 |
| 2.4 — bitwise tries, routing | **Nilsson & Karlsson**, *IP-address lookup using LC-tries*, 1999 | 🔍 |
| 3 — memory, and DAWGs | **Blumer et al.**, *The smallest automaton recognizing the subwords of a text*, 1985 | 🔍 |
| 3 — cache-conscious tries | **Askitis & Sinha**, HAT-trie, 2007 | ✅ |
| Q8 — persistent tries | **Bagwell**, *Ideal Hash Trees*, 2001 — the structure behind Clojure's collections | ✅ |

**Legend:** ✅ free at the link · 🔍 search the exact title on
[Google Scholar](https://scholar.google.com)

### If you read only one

**Askitis & Sinha on the HAT-trie**, read against §1.3.

§1.3 measures a trie's operation count as perfectly flat while its wall-clock time grows — the
textbook property holding exactly, and the machine disagreeing anyway. That gap is the paper's
entire subject. Their answer is not a better traversal: it is to **stop being a trie** below a
certain depth and become an array of small hash tables, so the pointer chase ends while the prefix
structure survives where it is actually used.

It is worth reading because it is a clean example of the move this series keeps arriving at — NB-04
§3's arrays over linked lists, NB-08 §1.6's B-trees over binary trees, NB-09 §1.4's 4-ary heaps.
Every one of them trades a theoretically clean pointer structure for a flatter one that the memory
hierarchy likes better, and every one of them was invented after somebody measured a structure whose
operation count was fine.

Then read **Fredkin's 1960 note** for the original idea in four pages, and **Morrison** for the
compression §1.5 implements.

***
# Appendix

| Symptom | Cause | Fix |
|---|---|---|
| `search` returns true for a prefix | Checked arrival, not the `is_word` flag (§1.2) | `node is not None and node.is_word` |
| `starts_with("")` true on an empty trie | The root always exists (§1.2) | Require a word at or below the node |
| Memory grows and never falls | `delete` clears the flag but does not prune (§1.2) | Prune back up; assert no dead subtrees |
| Trie uses far more memory than expected | Low prefix sharing — 6 nodes per word (§3) | Radix tree, or a hash set if prefixes are not needed |
| Radix insert corrupts the tree | The edge split, especially when the split point is the new word (§1.5) | Assert every edge label is non-empty and keyed by its first symbol |
| Array-node trie wastes memory | 26 slots per node, ~4% used (§1.4) | Map children, or a radix tree |
| Unicode words silently not found | NFC vs NFD — the same word takes two paths (§1.4, Q11) | Normalise at the boundary, once |
| Autocomplete ranking wrong on ties | Compound ordering hand-encoded as a tuple (§2.1) | Use a key function, not an encoding |
| Prefix query slower than expected | Rebuilding each result string from its path (§2.1) | Store the word at its node, or use a sorted array if static |
| Wildcard query is $O(n)$ | Leading `.` branches into every child (Practice 6) | Expected; do not claim the usual bound |
| Max-XOR trie gives wrong answers | Bits inserted least-significant first (§2.4) | Most significant first; the greedy proof depends on it |
| Trie slower than a hash set | It is, for exact lookup, at every size (§1.3) | Use a hash set unless you need prefixes |

## Checklist for trie code

- [ ] Does `search` check the **flag**, not just that the walk arrived (§1.2)?
- [ ] Does `delete` **prune**, and is that asserted structurally (§1.2)?
- [ ] Is `starts_with` correct on the empty prefix and the empty trie (§1.2)?
- [ ] What is a "symbol" — byte, code point, grapheme — and is input **normalised** (§1.4, Q11)?
- [ ] Array or map children, and does the alphabet justify the choice (§1.4)?
- [ ] Has the memory been measured against a plain `set` on **your** data (§3)?
- [ ] Is prefix sharing high enough to justify the structure (§3)?
- [ ] Is the dictionary static — would a **sorted array** be smaller and faster (§2.1)?
- [ ] Is any compound ordering built by hand instead of with a key function (§2.1)?
- [ ] Is a trie needed at all, or would a hash set answer the actual question (§1.3, Q12)?

## Where to go next

| Notebook | Why it follows |
|---|---|
| `dsu_zero_to_hero.ipynb` | Another structure whose whole value is one operation done extremely well |
| `range_queries_zero_to_hero.ipynb` | Indexing by position rather than by key structure |
| [`hashing_zero_to_hero.ipynb`](hashing_zero_to_hero.ipynb) | The strategy that wins on exact lookup and cannot do prefixes |
| [`strings_zero_to_hero.ipynb`](strings_zero_to_hero.ipynb) | §1.4's normalisation warning, and Aho-Corasick as §2.2 generalised |

See [`README.md`](README.md) for the full roster and reading order.